# HNFAS – final revision pipeline (no clustering; expert-based risk categories)

Run the cells **in order**. Every cell finds the dataset by name in Google Drive (`diabetes_dataset updated`), shows its tables and figures **in the cell output**, and saves them (figures at **400 dpi**) in `My Drive/HNFAS_results/`.

**Design.** Four **expert-based** risk categories (ADA HbA1c thresholds 5.7 %, 6.5 % and a conventional 8.0 % level); **no clustering anywhere**; HbA1c excluded from all inputs; the network input is the **18 principal components**; the WBM score is never a network input. Settings used for the manuscript: seed 42, batch 256, at most 60 epochs, fixed hyper-parameters.

| Cell | Produces | Manuscript items |
|---|---|---|
| 2–3 | pipeline; WBM validity; Stage-1 coefficient sensitivity | Section on coefficient sensitivity, Fig. 16 |
| 4 | manuscript experiments: overall/per-class metrics, ablation, baselines, 5-fold CV, corrected tests | Tables 7–10, 12; Figs 7–9, 14 |
| 5 | clinical validation of the categories | Fig. 13 |
| 6 | SHAP | Figs 10–11 |
| 7 | feature-importance agreement across models | Fig. 12 |
| 8 | (p,q) sensitivity and complexity table | Fig. 15, Table 10a |

In [ ]:
# 1) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

## 2) Core pipeline (defines functions only; nothing runs yet)

In [ ]:
"""
HNFAS -- OPTION 2 (independent clinical labels)  |  Colab-ready
================================================================
Why this version exists
-----------------------
In the original design the four risk classes were thresholds of the sigmoid-WBM
score, and that same score was fed to the network -> the label was a function of
an input (circular).  Here the labels are defined by an EXTERNAL clinical rule
that does not use the fuzzy/WBM pipeline at all, and the label-defining variable
is REMOVED from the model inputs.  The WBM score is then a genuine candidate
feature whose added value is tested, not assumed.

Clinical label rule (HbA1c, ADA Standards of Care diagnostic categories)
    0 Low        HbA1c < 5.7 %            (normoglycaemia)
    1 Moderate   5.7 - 6.4 %              (prediabetes range)
    2 High       6.5 - 7.9 %              (diabetes range)
    3 Very High  >= 8.0 %                 (diabetes range, markedly elevated)
 The 5.7 and 6.5 cut-points are the ADA diagnostic thresholds.  The 8.0 cut-point
 is a conventional "well above goal" level, NOT a diagnostic criterion -> it must
 be confirmed by a clinician and is a single parameter (CUTS) below.

Input policies (both are run and reported)
    A : drop only the label variable (hba1c)
    B : also drop the other glycaemic variables (glucose_fasting, glucose_postprandial)
        -> risk from clinical/lifestyle covariates only

Fuzzy pipeline (unchanged from the manuscript): PCA -> 5 Gaussian MFs ->
Stage-1 weights wc=(0.05,0.15,0.25,0.30,0.25) -> Weighted Bonferroni Mean ->
sigmoid(x0=0.3, kappa=8).  Everything is fitted on the TRAINING split only.
No clustering is used anywhere in the pipeline.
"""
import argparse, glob, json, os, pathlib, random, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, chi2
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score,
                             matthews_corrcoef, cohen_kappa_score, confusion_matrix)
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
LABEL_COL, CUTS = "hba1c", (5.7, 6.5, 8.0)
CLASS_NAMES = ["Low (<5.7)", "Moderate (5.7-6.4)", "High (6.5-7.9)", "Very High (>=8.0)"]
POLICIES = {"A": ["hba1c"],                                   # expert (ADA) labels; only the label variable removed from the inputs
            "B": ["hba1c", "glucose_fasting", "glucose_postprandial"]}   # expert (ADA) labels; all glycaemic variables removed (strict)
OUTCOME_COL = "diagnosed_diabetes"            # never used as an input
ID_COLUMNS = ("S.NO", "ID", "id", "patient_id", "diabetes_risk_score")   # identifiers + pre-computed score (not one of the 22 manuscript features)

N_COMPONENTS = 18
CENTRE_OFFSETS = np.array([-2.0, -0.8, 0.2, 1.2, 2.2])
WIDTH_FACTOR = 0.7
WC = np.array([0.05, 0.15, 0.25, 0.30, 0.25])      # Very High (0.25) < High (0.30)
BONF_P, BONF_Q = 1.0, 1.0
SIG_X0, SIG_KAPPA = 0.3, 8.0
LEVELS = ["VL", "L", "M", "H", "VH"]


def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)


class FuzzyWBM:
    """PCA scores -> memberships -> Stage 1 -> WBM -> sigmoid.  Fit on train only."""

    def __init__(self, wc=WC, p=BONF_P, q=BONF_Q, x0=SIG_X0, kappa=SIG_KAPPA):
        wc = np.asarray(wc, dtype=float)
        assert wc.shape == (5,) and np.all(wc >= 0) and abs(wc.sum() - 1) < 1e-9, \
            "Stage-1 coefficients must be non-negative and sum to 1 (R_im in [0,1])"
        self.wc, self.p, self.q, self.x0, self.kappa = wc, p, q, x0, kappa

    def fit(self, P_train, pca_eigvals):
        self.mu_ = P_train.mean(0)
        self.sd_ = P_train.std(0)
        self.w_ = pca_eigvals / pca_eigvals.sum()                 # Eq. 9, sums to 1
        K = len(self.w_)
        C = np.outer(self.w_, self.w_) / (1.0 - self.w_)[:, None]  # w_m w_r / (1-w_m)
        np.fill_diagonal(C, 0.0)                                   # m != r
        self.C_ = C
        assert abs(C.sum() - 1.0) < 1e-9                           # Eq. 14
        return self

    def memberships(self, P):
        c = self.mu_[:, None] + CENTRE_OFFSETS[None, :] * self.sd_[:, None]   # (K,5)
        tau = WIDTH_FACTOR * self.sd_[:, None]
        return np.exp(-0.5 * ((P[:, :, None] - c[None]) / tau[None]) ** 2)   # (N,K,5)

    def stage1(self, P):
        return (self.memberships(P) * self.wc[None, None, :]).sum(axis=2)     # (N,K)  Eq. 12

    def wbm(self, R):
        A, B = R ** self.p, R ** self.q
        num = (A * (B @ self.C_.T)).sum(axis=1)                               # Eq. 13
        return num ** (1.0 / (self.p + self.q))

    def raw_score(self, P):
        return self.wbm(self.stage1(P))

    def sigmoid(self, S):
        return 1.0 / (1.0 + np.exp(-self.kappa * (S - self.x0)))              # Eq. 15

    def transform(self, P):
        return self.sigmoid(self.raw_score(P))



def clinical_labels(hba1c):
    return np.digitize(np.asarray(hba1c, float), CUTS)


def with_weights(fz, wc):
    """Copy of a fitted FuzzyWBM using different Stage-1 weights (all other parameters unchanged)."""
    new = FuzzyWBM(wc)
    new.mu_, new.sd_, new.w_, new.C_ = fz.mu_, fz.sd_, fz.w_, fz.C_
    return new


# ----------------------------------------------------------------------------
def prepare(path, policy, extra_drop=()):
    """Labels = expert-based ADA HbA1c cut-points (label variable removed from the inputs).
       The network input is the 18 PCs (the WBM score is NOT a network input). No clustering is used anywhere."""
    df = load_table(path)
    df = df.drop(columns=[c for c in ID_COLUMNS if c in df.columns])
    lab_ada = clinical_labels(df[LABEL_COL].values)
    dx = df[OUTCOME_COL].values.astype(int) if OUTCOME_COL in df else None
    drop = [c for c in POLICIES[policy] + list(extra_drop) + [OUTCOME_COL] if c in df.columns]
    Xdf = df.drop(columns=drop)
    split_on = lab_ada

    trv, te = next(StratifiedShuffleSplit(1, test_size=0.20, random_state=SEED).split(Xdf, split_on))
    tr_i, va_i = next(StratifiedShuffleSplit(1, test_size=0.25, random_state=SEED).split(Xdf.iloc[trv], split_on[trv]))
    tr, va = trv[tr_i], trv[va_i]

    X = Xdf.fillna(Xdf.iloc[tr].median())
    X = X.loc[:, X.iloc[tr].var().values >= 1e-8]
    scaler = StandardScaler().fit(X.iloc[tr]); Z = scaler.transform(X)
    K = min(N_COMPONENTS, Z.shape[1])
    pca = PCA(n_components=K, random_state=SEED).fit(Z[tr]); P = pca.transform(Z)
    fz = FuzzyWBM().fit(P[tr], pca.explained_variance_)
    s = fz.transform(P)
    lab, thr = lab_ada, None
    info = dict(policy=policy, n_features=X.shape[1], features=list(X.columns), n_pcs=K,
                dropped=drop, pca_cum_var=float(pca.explained_variance_ratio_.sum()),
                label_rule=f"expert-based ADA HbA1c cut-points {CUTS}",
                split=(len(tr), len(va), len(te)),
                counts_all=np.bincount(lab, minlength=4).tolist(),
                counts_test=np.bincount(lab[te], minlength=4).tolist())
    return dict(P=P, s=s, lab=lab, lab_ada=lab_ada, dx=dx, thr=thr, tr=tr, va=va, te=te, fz=fz, pca=pca,
                X=X, scaler=scaler, info=info)


# ----------------------------------------------------------------------------
# Validity of the WBM score against the CLINICAL label, as a function of weights
# ----------------------------------------------------------------------------
def _validity(D, wc):
    te = D["te"]; P = D["P"][te]; lab = D["lab_ada"][te]
    raw = with_weights(D["fz"], wc).raw_score(P)
    base = D["fz"].raw_score(P)
    return dict(rho_label=float(spearmanr(raw, lab)[0]),
                auc_diabetes_range=float(roc_auc_score(lab >= 2, raw)),
                rho_vs_baseline=float(spearmanr(raw, base)[0]))


def run_sensitivity(D, n_random=100):
    rng = np.random.RandomState(SEED)
    named = {"Baseline\n(0.05,0.15,0.25,0.30,0.25)": WC,
             "Monotone\n(0.05,0.10,0.20,0.30,0.35)": np.array([0.05, 0.10, 0.20, 0.30, 0.35]),
             "Rank-sum\n(1..5)/15": np.arange(1, 6) / 15,
             "Uniform\n(0.20 each)": np.full(5, 0.2)}
    vec = pd.DataFrame([dict(vector=k.replace("\n", " "), **{f"w_{l}": float(x) for l, x in zip(LEVELS, v)},
                             **_validity(D, v)) for k, v in named.items()])
    rnd = []
    for i in range(n_random):
        wc = WC * rng.uniform(0.8, 1.2, 5); wc /= wc.sum()
        rnd.append(dict(draw=i + 1, **{f"w_{l}": float(x) for l, x in zip(LEVELS, wc)}, **_validity(D, wc)))
    oat = []
    for j, lvl in enumerate(LEVELS):
        for d in (-0.20, -0.10, 0.10, 0.20):
            wc = WC.copy(); wc[j] *= (1 + d); wc /= wc.sum()
            oat.append(dict(category=lvl, change=d * 100, **_validity(D, wc)))
    return dict(named=named, vec=vec, rnd=pd.DataFrame(rnd), oat=pd.DataFrame(oat))


# ----------------------------------------------------------------------------
# Classifier (manuscript Table 3)
# ----------------------------------------------------------------------------
def build_and_train(Xtr, ytr, Xva, yva, seed, max_epochs, batch_size, l2=1e-3, lr=1e-3):
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers, regularizers, callbacks
    keras.utils.set_random_seed(seed)
    reg = regularizers.l2(l2)
    m = keras.Sequential([
        layers.Input(shape=(Xtr.shape[1],)),
        layers.Dense(128, activation="relu", kernel_regularizer=reg, kernel_initializer="he_normal"),
        layers.BatchNormalization(), layers.Dropout(0.4),
        layers.Dense(64, activation="relu", kernel_regularizer=reg, kernel_initializer="he_normal"),
        layers.BatchNormalization(), layers.Dropout(0.3),
        layers.Dense(32, activation="relu", kernel_initializer="he_normal"),
        layers.BatchNormalization(), layers.Dropout(0.2),
        layers.Dense(4, activation="softmax")])
    m.compile(optimizer=keras.optimizers.Adam(lr), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    cw = compute_class_weight("balanced", classes=np.arange(4), y=ytr)
    m.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=max_epochs, batch_size=batch_size, verbose=0,
          class_weight={i: float(v) for i, v in enumerate(cw)},
          callbacks=[callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True),
                     callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=7, min_lr=1e-6)])
    return m


def metrics(y, pred, prob):
    return dict(accuracy=accuracy_score(y, pred), balanced_accuracy=balanced_accuracy_score(y, pred),
                f1_macro=f1_score(y, pred, average="macro"), f1_weighted=f1_score(y, pred, average="weighted"),
                auc_ovr=roc_auc_score(y, prob, multi_class="ovr", average="weighted", labels=range(4)),
                mcc=matthews_corrcoef(y, pred), kappa=cohen_kappa_score(y, pred))


def fit_predict(F, D, seed, max_epochs, batch_size):
    tr, va, te, y = D["tr"], D["va"], D["te"], D["lab"]
    sc = StandardScaler().fit(F[tr])
    m = build_and_train(sc.transform(F[tr]), y[tr], sc.transform(F[va]), y[va], seed, max_epochs, batch_size)
    return m.predict(sc.transform(F[te]), verbose=0)


def evaluate_models(D, seeds, max_epochs, batch_size):
    """PCs only vs PCs + WBM, several seeds, same split -> paired comparison."""
    F_pc = D["P"]; F_hy = np.hstack([D["P"], D["s"][:, None]]); y = D["lab"][D["te"]]
    rows, preds = [], {}
    for sd in seeds:
        for name, F in (("PCs only", F_pc), ("PCs + WBM", F_hy)):
            prob = fit_predict(F, D, sd, max_epochs, batch_size)
            rows.append(dict(model=name, seed=sd, **metrics(y, prob.argmax(1), prob)))
            preds[(name, sd)] = prob.argmax(1)
            print(f"   seed {sd} | {name:10s} | acc={rows[-1]['accuracy']:.4f} macroF1={rows[-1]['f1_macro']:.4f}")
    per_seed = pd.DataFrame(rows)
    mcols = [c for c in per_seed.columns if c not in ("model", "seed")]
    summ = per_seed.groupby("model")[mcols].agg(["mean", "std"])
    summ.columns = [f"{a}_{b}" for a, b in summ.columns]
    wide = per_seed.pivot(index="seed", columns="model", values=mcols)
    diff = pd.DataFrame({m: [(wide[(m, "PCs + WBM")] - wide[(m, "PCs only")]).mean(),
                             (wide[(m, "PCs + WBM")] - wide[(m, "PCs only")]).std()] for m in mcols},
                        index=["mean_diff (WBM - PCs)", "sd_diff"]).T.reset_index().rename(columns={"index": "metric"})
    a, b = preds[("PCs only", seeds[0])] == y, preds[("PCs + WBM", seeds[0])] == y
    n01, n10 = int((a & ~b).sum()), int((~a & b).sum())
    stat = (abs(n01 - n10) - 1) ** 2 / max(n01 + n10, 1)
    mcn = pd.DataFrame([dict(seed=seeds[0], PCs_only_right_WBM_wrong=n01, PCs_only_wrong_WBM_right=n10,
                             chi2=stat, p_value=float(chi2.sf(stat, 1)))])
    cm = pd.DataFrame(confusion_matrix(y, preds[("PCs + WBM", seeds[0])], labels=range(4)),
                      index=[f"true {c}" for c in CLASS_NAMES], columns=[f"pred {c}" for c in CLASS_NAMES])
    return dict(per_seed=per_seed, summary=summ.reset_index(), diff=diff, mcnemar=mcn, confusion=cm)


# ----------------------------------------------------------------------------
# Figures (400 dpi)
# ----------------------------------------------------------------------------
DPI = 400
COL = ["#1f77b4", "#2ca02c", "#ff7f0e", "#9467bd"]


def make_figures(R, fig_dir):
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False, "savefig.dpi": DPI})
    fig_dir = pathlib.Path(fig_dir); fig_dir.mkdir(parents=True, exist_ok=True); paths = []
    pols = list(R)

    def save(fig, name):
        p = fig_dir / name; fig.savefig(p, dpi=DPI, bbox_inches="tight"); plt.close(fig); paths.append(p)

    # Fig 1 weight vectors
    S0 = R[pols[0]]["S"]; named, rnd = S0["named"], S0["rnd"]
    fig, ax = plt.subplots(figsize=(7.5, 4.2)); n = len(named); w = 0.8 / n
    for i, (k, v) in enumerate(named.items()):
        ax.bar(np.arange(5) + i * w - 0.4 + w / 2, v, w, label=k.replace("\n", " "), color=COL[i], edgecolor="black", lw=0.5)
    lo, hi = rnd[[f"w_{l}" for l in LEVELS]].min().values, rnd[[f"w_{l}" for l in LEVELS]].max().values
    ax.errorbar(np.arange(5) - 0.4 + w / 2, WC, yerr=[np.clip(WC - lo, 0, None), np.clip(hi - WC, 0, None)], fmt="none", ecolor="black", capsize=3, lw=1,
                label="Baseline: range of random ±20% draws")
    ax.set_xticks(range(5)); ax.set_xticklabels(["Very Low", "Low", "Moderate", "High", "Very High"])
    ax.set_ylabel("Category weight $w^c_j$"); ax.set_xlabel("Linguistic risk category")
    ax.set_title("Stage-1 category weight vectors compared"); ax.legend(fontsize=8, frameon=False)
    save(fig, "Fig1_category_weight_vectors.png")

    # Fig 2 validity of the WBM score by weight vector
    fig, axs = plt.subplots(1, 2, figsize=(10.5, 4.2))
    for ax, key, yl, ttl in [(axs[0], "rho_label", "Spearman ρ (WBM score vs clinical class)", "(a) Ordinal association"),
                             (axs[1], "auc_diabetes_range", "AUC for HbA1c ≥ 6.5% (diabetes range)", "(b) Discrimination")]:
        wd = 0.38
        for pi, pol in enumerate(pols):
            v, r = R[pol]["S"]["vec"], R[pol]["S"]["rnd"][key]
            vals = list(v[key]) + [r.mean()]
            x = np.arange(5) + (pi - 0.5) * wd
            ax.bar(x, vals, wd, label=f"Policy {pol}", color=["#4c78a8", "#f58518"][pi], edgecolor="black", lw=0.5)
            ax.errorbar(4 + (pi - 0.5) * wd, r.mean(), yerr=[[max(0.0, r.mean() - r.quantile(.05))], [max(0.0, r.quantile(.95) - r.mean())]],
                        fmt="none", ecolor="black", capsize=3)
        ax.set_xticks(range(5)); ax.set_xticklabels(["Baseline", "Monotone", "Rank-sum", "Uniform", "Random ±20%\n(mean, 5–95%)"], fontsize=8)
        ax.set_ylabel(yl); ax.set_title(ttl)
        if key == "auc_diabetes_range": ax.axhline(0.5, color="grey", ls=":", lw=1); ax.set_ylim(0.4, ax.get_ylim()[1] * 1.06)
        else: ax.set_ylim(0, ax.get_ylim()[1] * 1.12)
        ax.legend(frameon=False, fontsize=8, loc="upper center", ncol=2)
    save(fig, "Fig2_validity_by_weight_vector.png")

    # Fig 3 random perturbations
    fig, axs = plt.subplots(1, len(pols), figsize=(5.2 * len(pols), 3.8), squeeze=False)
    for ax, pol in zip(axs[0], pols):
        r = R[pol]["S"]["rnd"]; b = R[pol]["S"]["vec"].rho_label.iloc[0]
        ax.hist(r.rho_label, bins=20, color="#4c78a8", edgecolor="black", lw=0.5)
        ax.axvline(b, color="red", ls="--", label=f"baseline = {b:.3f}")
        ax.set_xlabel("Spearman ρ (WBM vs clinical class)"); ax.set_ylabel("Number of draws")
        ax.set_title(f"Policy {pol}: {len(r)} random ±20% draws"); ax.legend(frameon=False)
    save(fig, "Fig3_random_perturbation_distribution.png")

    # Fig 4 one-category-at-a-time
    fig, axs = plt.subplots(1, len(pols), figsize=(5.4 * len(pols), 4.0), squeeze=False)
    for ax, pol in zip(axs[0], pols):
        o = R[pol]["S"]["oat"]; b = R[pol]["S"]["vec"].rho_label.iloc[0]
        for lvl in LEVELS:
            d = o[o.category == lvl].sort_values("change")
            ax.plot(np.insert(d.change.values, 2, 0.0), np.insert(d.rho_label.values, 2, b), marker="o", label=lvl)
        ax.set_xlabel("Relative change of one weight (%)"); ax.set_ylabel("Spearman ρ (WBM vs clinical class)")
        ax.set_title(f"Policy {pol}: one weight at a time"); ax.legend(title="Weight", frameon=False, ncol=2)
    save(fig, "Fig4_one_category_at_a_time.png")

    # Fig 5 WBM score by clinical class
    fig, axs = plt.subplots(1, len(pols), figsize=(5.4 * len(pols), 4.0), squeeze=False)
    for ax, pol in zip(axs[0], pols):
        D = R[pol]["D"]; te = D["te"]
        ax.boxplot([D["s"][te][D["lab_ada"][te] == c] for c in range(4)], showfliers=False, patch_artist=True,
                   boxprops=dict(facecolor="#c6dbef"), medianprops=dict(color="black"))
        ax.set_xticklabels(["Low", "Moderate", "High", "V. High"]); ax.set_xlabel("Clinical class (HbA1c)")
        ax.set_ylabel("Sigmoid-WBM score (test)"); ax.set_title(f"Policy {pol}")
    save(fig, "Fig5_WBM_score_by_clinical_class.png")

    # Fig 6 model comparison
    if all("M" in R[p] for p in pols):
        fig, axs = plt.subplots(1, len(pols), figsize=(5.4 * len(pols), 4.0), squeeze=False)
        for ax, pol in zip(axs[0], pols):
            s = R[pol]["M"]["summary"].set_index("model"); x = np.arange(3); wd = 0.38
            for i, m in enumerate(["PCs only", "PCs + WBM"]):
                ax.bar(x + (i - 0.5) * wd, [s.loc[m, f"{k}_mean"] for k in ("accuracy", "balanced_accuracy", "f1_macro")], wd,
                       yerr=[s.loc[m, f"{k}_std"] for k in ("accuracy", "balanced_accuracy", "f1_macro")],
                       capsize=3, label=m, edgecolor="black", lw=0.5, color=["#9ecae1", "#3182bd"][i])
            ct = R[pol]["D"]["info"]["counts_test"]
            ax.hlines(max(ct) / sum(ct), -0.5, 0.5, color="grey", ls=":", lw=1.5, label="Majority-class accuracy")
            ax.hlines(0.25, 0.5, 2.5, color="black", ls=":", lw=1.2, label="Chance (4 balanced classes)")
            ax.set_xticks(x); ax.set_xticklabels(["Accuracy", "Balanced acc.", "Macro-F1"]); ax.set_ylim(0, 1.35)
            ax.set_title(f"Policy {pol} (mean ± SD over seeds)"); ax.legend(frameon=False, fontsize=7.5, loc="upper center", ncol=2)
        save(fig, "Fig6_PCs_vs_PCs_plus_WBM.png")
    return paths


# ----------------------------------------------------------------------------
# Drive / Excel
# ----------------------------------------------------------------------------
DATA_STEM = "diabetes_dataset updated"          # your file: "diabetes_dataset updated.csv" (matching ignores case, spaces, _ and -)
DRIVE_OUT_DIR = "/content/drive/MyDrive/HNFAS_results"
RESULTS_FILE = "HNFAS_option2_results.xlsx"


def _norm(name):
    """'diabetes_dataset updated.csv' == 'Diabetes-Dataset_Updated (1).CSV' for matching purposes."""
    import re
    return re.sub(r"[^a-z0-9]", "", os.path.splitext(name)[0].lower())


def resolve_data_path(path=None, stem=DATA_STEM, roots=None, max_depth=5):
    """Find the dataset: explicit path -> name search in Drive (tolerant to spaces/underscores) -> upload dialog."""
    if path and os.path.exists(path):
        return path
    roots = roots or ["/content/drive/MyDrive", "/content", os.getcwd()]
    ok = (".csv", ".xlsx", ".xls", ".gsheet")
    want = _norm(stem); hits, seen = [], []
    for r in roots:
        if not os.path.isdir(r): continue
        base_depth = r.rstrip(os.sep).count(os.sep)
        for dp, dn, fn in os.walk(r):
            if dp.count(os.sep) - base_depth >= max_depth: dn[:] = []
            dn[:] = [d for d in dn if not d.startswith(".") and d not in ("proc", "sys", "sample_data")]
            for f in fn:
                if f.lower().endswith(ok):
                    seen.append(os.path.join(dp, f))
                    if _norm(f).startswith(want): hits.append(os.path.join(dp, f))
    if hits:
        hits = sorted(set(hits), key=lambda f: (f.lower().endswith(".gsheet"), f.count(os.sep), len(f)))
        print("Dataset found:", hits[0])
        if len(hits) > 1: print("  (other matches ignored:", *hits[1:4], ")")
        return hits[0]
    print(f"No file matching '{stem}' found. Data-like files seen in Drive/session:")
    for f in seen[:15]: print("   ", f)
    try:                                                        # last resort: upload dialog
        from google.colab import files
        print("Please select the dataset file now ...")
        up = files.upload()
        if up: return next(iter(up))
    except ImportError:
        pass
    raise FileNotFoundError(f"Could not find '{stem}'. Set DATA_PATH in the settings cell to the file's full path.")


def load_gsheet(gsheet_path):
    """Read a Google Sheet (Drive shows it WITHOUT an extension; Colab sees it as .gsheet)."""
    import json
    try:
        from google.colab import auth; auth.authenticate_user()
        import gspread
        from google.auth import default
        creds, _ = default(); gc = gspread.authorize(creds)
        key = json.load(open(gsheet_path))["doc_id"]
        vals = gc.open_by_key(key).sheet1.get_all_values()
    except Exception as e:
        raise RuntimeError(f"Could not read the Google Sheet directly ({e}). Workaround: in Drive open the sheet, "
                           "File > Download > CSV, upload that CSV to My Drive, and run again.")
    df = pd.DataFrame(vals[1:], columns=vals[0])
    for c in df.columns:
        try: df[c] = pd.to_numeric(df[c])
        except (ValueError, TypeError): pass
    return df


def load_table(path):
    pl = str(path).lower()
    if pl.endswith(".gsheet"): return load_gsheet(path)
    if pl.endswith((".xlsx", ".xls")): return pd.read_excel(path)
    return pd.read_csv(path)


def write_workbook(path, R, fig_paths):
    with pd.ExcelWriter(path, engine="openpyxl") as xw:
        rows = [("Label rule", "HbA1c cut-points " + str(CUTS) + " (5.7/6.5 = ADA; 8.0 = conventional, needs clinical sign-off)"),
                ("Stage-1 weights", str(list(WC))), ("Outcome column (never an input)", OUTCOME_COL)]
        for p, r in R.items():
            i = r["D"]["info"]
            rows += [(f"Policy {p}: dropped inputs", ", ".join(i["dropped"])), (f"Policy {p}: features / PCs", f"{i['n_features']} / {i['n_pcs']}"),
                     (f"Policy {p}: label rule", i["label_rule"]), (f"Policy {p}: PCA cumulative variance", round(i["pca_cum_var"], 4)),
                     (f"Policy {p}: class counts (all)", str(i["counts_all"])), (f"Policy {p}: train/val/test", str(i["split"]))]
        pd.DataFrame(rows, columns=["item", "value"]).to_excel(xw, sheet_name="Summary", index=False)
        for p, r in R.items():
            r["S"]["vec"].to_excel(xw, sheet_name=f"Sens_vectors_{p}", index=False)
            r["S"]["rnd"].to_excel(xw, sheet_name=f"Sens_random_{p}", index=False)
            r["S"]["oat"].to_excel(xw, sheet_name=f"Sens_one_at_a_time_{p}", index=False)
            if "M" in r:
                r["M"]["per_seed"].to_excel(xw, sheet_name=f"Models_per_seed_{p}", index=False)
                r["M"]["summary"].to_excel(xw, sheet_name=f"Models_summary_{p}", index=False)
                r["M"]["diff"].to_excel(xw, sheet_name=f"Paired_diff_{p}", index=False)
                r["M"]["mcnemar"].to_excel(xw, sheet_name=f"McNemar_{p}", index=False)
                r["M"]["confusion"].to_excel(xw, sheet_name=f"Confusion_{p}")
        try:
            from openpyxl.drawing.image import Image as XLImage
            ws = xw.book.create_sheet("Figures"); row = 1
            for fp in fig_paths:
                img = XLImage(str(fp)); k = 900 / img.width; img.width, img.height = 900, int(img.height * k)
                ws.add_image(img, f"A{row}"); row += int(img.height / 20) + 3
        except Exception as e:
            print("Figures not embedded:", e)


def _show(obj, title=None):
    """Print in a terminal, render as a table in Colab/Jupyter."""
    try:
        from IPython.display import display, Markdown
        if title: display(Markdown(f"**{title}**"))
        display(obj)
    except ImportError:
        if title: print("\n" + title)
        print(obj.to_string() if hasattr(obj, "to_string") else obj)


def _show_figs(paths, width=900):
    try:
        from IPython.display import display, Image
        for p in paths:
            print(pathlib.Path(p).name); display(Image(filename=str(p), width=width))
    except ImportError:
        pass


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default=None, help="full path (optional; otherwise searched by name)")
    ap.add_argument("--data-stem", default=DATA_STEM)
    ap.add_argument("--out-dir", default=DRIVE_OUT_DIR)
    ap.add_argument("--policies", default="A,B")
    ap.add_argument("--seeds", type=int, default=3)
    ap.add_argument("--max-epochs", type=int, default=60)
    ap.add_argument("--batch-size", type=int, default=256)
    ap.add_argument("--n-random", type=int, default=100)
    ap.add_argument("--skip-nn", action="store_true")
    ap.add_argument("--no-drive", action="store_true")
    a, _ = ap.parse_known_args()
    if not a.no_drive:
        try:
            from google.colab import drive; drive.mount("/content/drive")
        except ImportError:
            pass
    seed_everything()
    data = resolve_data_path(a.data, a.data_stem)
    try:
        out = pathlib.Path(a.out_dir); out.mkdir(parents=True, exist_ok=True)
    except Exception as e:
        print("Could not write to Drive folder (", e, ") -> using /content/HNFAS_results"); out = pathlib.Path("/content/HNFAS_results"); out.mkdir(parents=True, exist_ok=True)
    print(f"\nLabel rule: HbA1c cut-points {CUTS}  |  Stage-1 weights {[float(x) for x in WC]}")

    # ---- PHASE 1: data, fuzzy score, weight sensitivity (fast) ----------------------
    R = {}
    for pol in a.policies.split(","):
        D = prepare(data, pol); i = D["info"]
        print(f"\n===== Policy {pol}: inputs exclude {POLICIES[pol]} =====")
        print(f"{i['n_features']} features -> {i['n_pcs']} PCs (cum. variance {i['pca_cum_var']:.3f}); "
              f"class counts Low/Mod/High/VHigh = {i['counts_all']}")
        S = run_sensitivity(D, a.n_random); R[pol] = dict(D=D, S=S)
        _show(S["vec"].round(4), f"Policy {pol} - WBM validity for each category-weight vector (test set)")
        _show(S["oat"].pivot(index="category", columns="change", values="rho_label").loc[LEVELS].round(4),
              f"Policy {pol} - Spearman rho when ONE weight changes by the given % (baseline rho = {S['vec'].rho_label.iloc[0]:.4f})")
        r = S["rnd"]
        print(f"Random ±20% draws (n={len(r)}): rho mean {r.rho_label.mean():.4f}, 5-95% [{r.rho_label.quantile(.05):.4f}, {r.rho_label.quantile(.95):.4f}]")
    print("\nFigures 1-5 (400 dpi):")
    figs = make_figures(R, out / "figures_400dpi"); _show_figs(figs)
    write_workbook(out / RESULTS_FILE, R, figs)                 # interim save
    print("Interim results saved:", out / RESULTS_FILE)

    # ---- PHASE 2: does the WBM feature help the network? (slower) ---------------------
    if not a.skip_nn:
        for pol in R:
            print(f"\n===== Policy {pol}: training PCs-only vs PCs+WBM ({a.seeds} seeds) =====")
            R[pol]["M"] = M = evaluate_models(R[pol]["D"], list(range(a.seeds)), a.max_epochs, a.batch_size)
            _show(M["summary"][["model", "accuracy_mean", "accuracy_std", "balanced_accuracy_mean", "balanced_accuracy_std",
                                "f1_macro_mean", "f1_macro_std", "auc_ovr_mean", "auc_ovr_std"]].round(4), f"Policy {pol} - test metrics (mean, SD over seeds)")
            _show(M["diff"].round(4), f"Policy {pol} - paired difference (PCs+WBM minus PCs only)")
            _show(M["mcnemar"].round(4), f"Policy {pol} - McNemar test (first seed)")
            _show(M["confusion"], f"Policy {pol} - confusion matrix, PCs+WBM (first seed)")
            d = M["diff"].set_index("metric").loc["balanced_accuracy"]
            m0, sd0 = float(d.iloc[0]), float(d.iloc[1])
            note = ("use >= 2 seeds to judge seed-to-seed noise" if not np.isfinite(sd0) else
                    "within seed-to-seed noise: no evidence of benefit" if abs(m0) <= 2 * sd0 else "larger than seed noise - inspect")
            print(f"-> Balanced-accuracy change from adding WBM: {m0:+.4f} (SD {sd0:.4f}); {note}.")
        figs = make_figures(R, out / "figures_400dpi")
        print("\nFigure 6 (400 dpi):"); _show_figs([f for f in figs if "Fig6" in f.name])
        write_workbook(out / RESULTS_FILE, R, figs)
    print("\nSaved single results file:", out / RESULTS_FILE)


## 3) Core analysis: WBM validity and Stage-1 coefficient sensitivity (`SKIP_NN = True` is enough for the manuscript's sensitivity results)

In [ ]:
import sys

DATA_NAME = 'diabetes_dataset updated'
DATA_PATH = None
OUT_DIR   = '/content/drive/MyDrive/HNFAS_results'

POLICIES_TO_RUN = 'A'      # 'A,B' adds the strict robustness setting without glucose variables
SKIP_NN = True
SEEDS = 3
N_RANDOM = 100

args = ['run', '--data-stem', DATA_NAME, '--out-dir', OUT_DIR, '--n-random', str(N_RANDOM), '--seeds', str(SEEDS), '--policies', POLICIES_TO_RUN]
if DATA_PATH: args += ['--data', DATA_PATH]
if SKIP_NN: args += ['--skip-nn']
sys.argv = args
main()

## 4) Manuscript experiments (Tables 7–10 and 12; Figures 7–9 and 14) — about 20–30 minutes

In [ ]:
# =====================================================================================
# CELL 4 -- the manuscript's experiments: Tables 7-10 and 12, Figures 7-9 and 14.        Run AFTER Cell 2.
#   * network input = 18 PCs (WBM score not an input); expert-based ADA labels; no clustering
#   * seed 42, batch 256, at most 60 epochs (early stopping, patience 15), fixed hyper-parameters
#   * baselines (LR, RF, XGBoost, SVM) on the same 18 PCs; SVM trained on a random subsample of 10,000 patients
#   * 5-fold stratified CV; preprocessing/PCA/fuzzy parameters re-estimated inside every training fold
#   * paired comparison: corrected resampled t-test (Nadeau & Bengio, 2003)
# =====================================================================================
import time, json
from math import sqrt
from scipy.stats import t as _tdist
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import precision_recall_fscore_support
try:
    from xgboost import XGBClassifier
except ImportError:
    raise ImportError("pip install xgboost")

MS_DATA_NAME = "diabetes_dataset updated"; MS_DATA_PATH = None
MS_OUT_DIR = "/content/drive/MyDrive/HNFAS_results"; MS_FILE = "HNFAS_manuscript_experiments.xlsx"
MS_EPOCHS, MS_BATCH, MS_SVMCAP, MS_FOLDS = 60, 256, 10000, 5
MS_FAST = False          # True = tiny smoke test (1 epoch, small baselines, 2 folds); results are NOT for the paper


def _full(y, prob):
    pred = prob.argmax(1); cm = confusion_matrix(y, pred, labels=range(4)); spec = []
    for c in range(4):
        tp = cm[c, c]; fn = cm[c].sum() - tp; fp = cm[:, c].sum() - tp; tn = cm.sum() - tp - fn - fp; spec.append(tn / (tn + fp))
    pw, rw, fw, _ = precision_recall_fscore_support(y, pred, average="weighted", zero_division=0)
    return dict(accuracy=accuracy_score(y, pred), balanced_accuracy=balanced_accuracy_score(y, pred), precision=pw, recall=rw,
                specificity=float(np.mean(spec)), f1=fw, f1_macro=f1_score(y, pred, average="macro"),
                auc=roc_auc_score(y, prob, multi_class="ovr", average="weighted", labels=range(4)),
                mcc=matthews_corrcoef(y, pred), kappa=cohen_kappa_score(y, pred))


def _nn(F, tr, va, te, y, seed=SEED):
    sc = StandardScaler().fit(F[tr]); ep = 1 if MS_FAST else MS_EPOCHS
    m = build_and_train(sc.transform(F[tr]), y[tr], sc.transform(F[va]), y[va], seed, ep, MS_BATCH)
    return m, m.predict(sc.transform(F[te]), verbose=0)


def _baselines(P, tr, te, y, seed=SEED):
    out = {}; sc = StandardScaler().fit(P[tr]); A, B = sc.transform(P[tr]), sc.transform(P[te]); sw = compute_sample_weight("balanced", y[tr])
    n_rf, n_xg, cap = (10, 10, 500) if MS_FAST else (200, 200, MS_SVMCAP)
    out["LR"] = LogisticRegression(max_iter=500, class_weight="balanced").fit(A, y[tr]).predict_proba(B)
    out["RF"] = RandomForestClassifier(n_estimators=n_rf, min_samples_leaf=5, class_weight="balanced_subsample", n_jobs=-1, random_state=seed).fit(A, y[tr]).predict_proba(B)
    out["XGBoost"] = XGBClassifier(n_estimators=n_xg, max_depth=6, learning_rate=0.1, n_jobs=-1, random_state=seed, verbosity=0).fit(A, y[tr], sample_weight=sw).predict_proba(B)
    idx = np.random.RandomState(seed).choice(tr, size=min(cap, len(tr)), replace=False); s2 = StandardScaler().fit(P[idx])
    out["SVM"] = SVC(kernel="rbf", C=10, gamma="scale", class_weight="balanced", probability=True, random_state=seed).fit(s2.transform(P[idx]), y[idx]).predict_proba(s2.transform(P[te]))
    return out


def run_manuscript_experiments():
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    seed_everything(); t0 = time.time(); path = resolve_data_path(MS_DATA_PATH, MS_DATA_NAME)
    out = pathlib.Path(MS_OUT_DIR)
    try: out.mkdir(parents=True, exist_ok=True)
    except Exception: out = pathlib.Path("/content/HNFAS_results"); out.mkdir(parents=True, exist_ok=True)
    fd = out / "figures_400dpi"; fd.mkdir(parents=True, exist_ok=True)
    plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
    # ------------------------- single test split: Tables 7-10, Figures 7-9
    D = prepare(path, "A"); P, s, y, tr, va, te = D["P"], D["s"], D["lab"], D["tr"], D["va"], D["te"]; yt = y[te]
    m, prob = _nn(P, tr, va, te, y); H = _full(yt, prob); hist = m.history.history; best = int(np.argmin(hist["val_loss"]) + 1)
    cm = confusion_matrix(yt, prob.argmax(1), labels=range(4)); pr, rc, f1, sup = precision_recall_fscore_support(yt, prob.argmax(1), labels=range(4))
    names = ["Low", "Moderate", "High", "Very High"]; N = int(sup.sum())
    t8 = pd.DataFrame({"class": names + ["Accuracy", "Macro avg", "Weighted avg"],
                       "precision": list(pr) + [H["accuracy"], pr.mean(), (pr * sup).sum() / N], "recall": list(rc) + [H["accuracy"], rc.mean(), (rc * sup).sum() / N],
                       "F1": list(f1) + [H["accuracy"], f1.mean(), (f1 * sup).sum() / N], "support": list(sup) + [N, N, N]})
    t7 = pd.DataFrame({"metric": ["Accuracy", "Weighted precision", "Weighted recall", "Weighted specificity (mean TNR)", "Weighted F1", "AUC-ROC (OvR weighted)", "MCC", "Cohen kappa", "Balanced accuracy"],
                       "value": [H[k] for k in ["accuracy", "precision", "recall", "specificity", "f1", "auc", "mcc", "kappa", "balanced_accuracy"]]})
    _show(t7.round(4), "Table 7 - test-set performance (network on 18 PCs)"); _show(t8.round(4), "Table 8 - per-class report")
    _, p1 = _nn(D["scaler"].transform(D["X"]), tr, va, te, y); A1 = _full(yt, p1)
    _, p3 = _nn(np.hstack([P, s[:, None]]), tr, va, te, y); A3 = _full(yt, p3)
    sc1 = StandardScaler().fit(s[tr].reshape(-1, 1)); lr1 = LogisticRegression(max_iter=500, class_weight="balanced").fit(sc1.transform(s[tr].reshape(-1, 1)), y[tr])
    A4 = _full(yt, lr1.predict_proba(sc1.transform(s[te].reshape(-1, 1))))
    keys = ["accuracy", "precision", "recall", "specificity", "f1", "auc", "mcc"]
    t9 = pd.DataFrame([{"configuration": n, **{k: d[k] for k in keys}} for n, d in [("A1: NN on standardized features", A1), ("A2: NN on 18 PCs (proposed)", H), ("A3: NN on 18 PCs + WBM score", A3), ("A4: LR on WBM score alone", A4)]])
    BL = {k: _full(yt, v) for k, v in _baselines(P, tr, te, y).items()}
    t10 = pd.DataFrame([{"model": n, **{k: d[k] for k in keys + ["kappa"]}} for n, d in [("Random Forest", BL["RF"]), ("XGBoost", BL["XGBoost"]), ("SVM", BL["SVM"]), ("Logistic Regression", BL["LR"]), ("Proposed HNFAS", H)]])
    _show(t9.round(4), "Table 9 - ablation"); _show(t10.round(4), "Table 10 - baselines on the same 18 PCs")
    ep = np.arange(1, len(hist["loss"]) + 1); fig, axs = plt.subplots(1, 2, figsize=(11, 4))
    for ax, (a_, b_, t_) in zip(axs, [("loss", "val_loss", "(a) Loss"), ("accuracy", "val_accuracy", "(b) Accuracy")]):
        ax.plot(ep, hist[a_], label="training"); ax.plot(ep, hist[b_], label="validation"); ax.axvline(best, color="grey", ls=":"); ax.set_xlabel("Epoch"); ax.legend(frameon=False); ax.set_title(t_)
    p7 = fd / "Fig7_training_curves.png"; fig.tight_layout(); fig.savefig(p7, dpi=400, bbox_inches="tight"); plt.close(fig)
    fig, axs = plt.subplots(1, 2, figsize=(11, 4.6))
    for ax, M_, t_, fm in [(axs[0], cm, "(a) Counts", "{:.0f}"), (axs[1], cm / cm.sum(1, keepdims=True) * 100, "(b) Row-normalised (%)", "{:.1f}")]:
        im = ax.imshow(M_, cmap="Blues"); ax.set_xticks(range(4)); ax.set_yticks(range(4)); ax.set_xticklabels(names, rotation=20); ax.set_yticklabels(names)
        for a in range(4):
            for c in range(4): ax.text(c, a, fm.format(M_[a, c]), ha="center", va="center", color="white" if M_[a, c] > M_.max() * .55 else "black", fontsize=9)
        ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(t_); fig.colorbar(im, ax=ax, fraction=.046)
    p8 = fd / "Fig8_confusion_matrix.png"; fig.tight_layout(); fig.savefig(p8, dpi=400, bbox_inches="tight"); plt.close(fig)
    cfg = [("A1: NN, standardized\nfeatures", A1), ("A2: NN, 18 PCs\n(proposed)", H), ("A3: NN, 18 PCs\n+ WBM score", A3), ("A4: LR, WBM\nscore alone", A4)]
    fig, ax = plt.subplots(figsize=(10, 4.4)); w = .2; x = np.arange(4)
    for i_, (k, l) in enumerate([("accuracy", "Accuracy"), ("f1", "F1-score"), ("auc", "AUC-ROC"), ("mcc", "MCC")]):
        v = [c[1][k] for c in cfg]; ax.bar(x + (i_ - 1.5) * w, v, w, label=l, edgecolor="k", lw=.4)
        for xx, vv in zip(x + (i_ - 1.5) * w, v): ax.text(xx, vv + .01, f"{vv:.2f}", ha="center", fontsize=7)
    ax.set_xticks(x); ax.set_xticklabels([c[0] for c in cfg], fontsize=8.5); ax.set_ylim(0, 1.08); ax.legend(frameon=False, ncol=4, loc="upper right", fontsize=8)
    p9 = fd / "Fig9_ablation.png"; fig.tight_layout(); fig.savefig(p9, dpi=400, bbox_inches="tight"); plt.close(fig)
    _show_figs([p7, p8, p9], width=850); print(f"single split done ({(time.time()-t0)/60:.1f} min); best epoch {best} of {len(ep)}")
    # ------------------------- 5-fold cross-validation: Figure 14, Table 12
    df = load_table(path); df = df.drop(columns=[c for c in ID_COLUMNS if c in df.columns]); lab = clinical_labels(df[LABEL_COL].values)
    Xdf = df.drop(columns=[c for c in POLICIES["A"] + [OUTCOME_COL] if c in df.columns]); rows = []; folds = 2 if MS_FAST else MS_FOLDS
    for k, (trf, tef) in enumerate(StratifiedKFold(folds, shuffle=True, random_state=SEED).split(Xdf, lab)):
        X = Xdf.fillna(Xdf.iloc[trf].median()); X = X.loc[:, X.iloc[trf].var().values >= 1e-8]; Z = StandardScaler().fit(X.iloc[trf]).transform(X)
        pca = PCA(18, random_state=SEED).fit(Z[trf]); Pf = pca.transform(Z); sf = FuzzyWBM().fit(Pf[trf], pca.explained_variance_).transform(Pf)
        i_tr, i_va = next(StratifiedShuffleSplit(1, test_size=0.15, random_state=SEED + k).split(Z[trf], lab[trf])); trn, val = trf[i_tr], trf[i_va]
        res = {}; _, res["HNFAS"] = _nn(Pf, trn, val, tef, lab, SEED + k); _, res["HNFAS+WBM"] = _nn(np.hstack([Pf, sf[:, None]]), trn, val, tef, lab, SEED + k)
        res.update(_baselines(Pf, trf, tef, lab, SEED + k))
        for nm_, pr_ in res.items(): rows.append(dict(fold=k, model=nm_, **_full(lab[tef], pr_)))
        print(f"  fold {k+1}/{folds} done ({(time.time()-t0)/60:.1f} min)")
    cvd = pd.DataFrame(rows); summ = cvd.groupby("model")[["accuracy", "balanced_accuracy", "f1_macro", "auc", "mcc"]].agg(["mean", "std"]); summ.columns = [f"{a}_{b}" for a, b in summ.columns]
    _show(summ.round(4), "5-fold cross-validated performance (mean, SD over folds)")

    def paired(a, b, metric="balanced_accuracy"):
        wv = cvd.pivot(index="fold", columns="model", values=metric); d = (wv[a] - wv[b]).values; n = len(d); mm = d.mean(); sd = d.std(ddof=1)
        se = sd * sqrt(1 / n + 1 / (folds - 1)); tc = _tdist.ppf(.975, n - 1)
        return dict(comparison=f"{a} minus {b}", metric=metric, mean_diff=mm, ci_low=mm - tc * se, ci_high=mm + tc * se, p_corrected=2 * _tdist.sf(abs(mm / se), n - 1) if se > 0 else np.nan)
    t12 = pd.DataFrame([paired("HNFAS", b) for b in ["RF", "XGBoost", "SVM", "LR"]] + [paired("HNFAS+WBM", "HNFAS", m_) for m_ in ["balanced_accuracy", "accuracy", "f1_macro", "auc", "mcc"]])
    _show(t12.round(4), "Table 12 - corrected paired t-tests (last five rows: effect of adding the WBM score)")
    order = ["LR", "RF", "XGBoost", "SVM", "HNFAS"]; lbl = ["Logistic\nRegression", "Random\nForest", "XGBoost", "SVM", "HNFAS"]; fig, axs = plt.subplots(1, 2, figsize=(11, 4.2))
    for ax, (met, tt) in zip(axs, [("balanced_accuracy", "(a) Balanced accuracy"), ("accuracy", "(b) Accuracy")]):
        data = [cvd[cvd.model == m_][met].values for m_ in order]; bp = ax.boxplot(data, patch_artist=True, widths=.55)
        for p_, c_ in zip(bp["boxes"], ["#9ecae1", "#a1d99b", "#fdae6b", "#d4b9da", "#d62728"]): p_.set_facecolor(c_)
        for i_, d in enumerate(data, 1): ax.scatter(np.full(len(d), i_), d, color="k", s=12, zorder=3)
        ax.set_xticklabels(lbl, fontsize=8.5); ax.set_ylabel(met.replace("_", " ").capitalize()); ax.set_title(tt)
    p14 = fd / "Fig14_cv_boxplots.png"; fig.tight_layout(); fig.savefig(p14, dpi=400, bbox_inches="tight"); plt.close(fig); _show_figs([p14], width=850)
    with pd.ExcelWriter(out / MS_FILE, engine="openpyxl") as xw:
        t7.to_excel(xw, sheet_name="Table7", index=False); t8.to_excel(xw, sheet_name="Table8", index=False); t9.to_excel(xw, sheet_name="Table9", index=False)
        t10.to_excel(xw, sheet_name="Table10", index=False); cvd.to_excel(xw, sheet_name="CV_per_fold", index=False); summ.reset_index().to_excel(xw, sheet_name="CV_summary", index=False)
        t12.to_excel(xw, sheet_name="Table12", index=False); pd.DataFrame(cm, index=names, columns=names).to_excel(xw, sheet_name="Confusion_matrix")
    print("Saved:", out / MS_FILE, f"| total {(time.time()-t0)/60:.1f} min")


run_manuscript_experiments()


## 5) Clinical validation of the four risk categories (Figure 13)

In [ ]:
# =====================================================================================
# CELL 4 -- clinical validation of the four risk categories  (run AFTER Cell 2)
# Output appears below the cell; tables + figure are saved in ONE Excel file.
# =====================================================================================
from scipy.stats import spearmanr
from sklearn.metrics import cohen_kappa_score

VAL_DATA_NAME = "diabetes_dataset updated"
VAL_DATA_PATH = None
VAL_OUT_DIR = "/content/drive/MyDrive/HNFAS_results"
VAL_FILE = "HNFAS_option2_clinical_validation.xlsx"


def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d; h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (c - h) * 100, (c + h) * 100


def profile(lab, df, name):
    dx, hb, fpg = df[OUTCOME_COL].values, df["hba1c"].values, df["glucose_fasting"].values
    rows = []
    for c, nm in enumerate(["Low", "Moderate", "High", "Very High"]):
        m = lab == c; n = int(m.sum()); k = int(dx[m].sum()); lo, hi = wilson(k, n)
        rows.append({"class": nm, "n": n, "% of cohort": 100 * n / len(lab), "mean HbA1c (%)": hb[m].mean(),
                     "mean fasting glucose (mg/dL)": fpg[m].mean(), "HbA1c >= 6.5% (%)": 100 * (hb[m] >= 6.5).mean(),
                     "diagnosed diabetes (%)": 100 * k / n, "95% CI low": lo, "95% CI high": hi})
    t = pd.DataFrame(rows)
    extra = {"Spearman (class, diagnosis)": spearmanr(lab, dx)[0], "Spearman (class, HbA1c)": spearmanr(lab, hb)[0],
             "Spearman (class, fasting glucose)": spearmanr(lab, fpg)[0]}
    return t, extra


def run_clinical_validation():
    seed_everything()
    data = resolve_data_path(VAL_DATA_PATH, VAL_DATA_NAME)
    df = load_table(data); df = df.drop(columns=[c for c in ID_COLUMNS if c in df.columns])
    lab = clinical_labels(df[LABEL_COL].values); out = pathlib.Path(VAL_OUT_DIR)
    try: out.mkdir(parents=True, exist_ok=True)
    except Exception: out = pathlib.Path("/content/HNFAS_results"); out.mkdir(parents=True, exist_ok=True)

    tab, ex = profile(lab, df, "clinical")
    _show(tab.round(2), "Table A - HbA1c-based (ADA) risk categories validated against the diagnosis and fasting glucose")
    _show(pd.DataFrame([ex]).round(3), "Monotonic association of the categories with external variables")

    fpg, hb = df["glucose_fasting"].values, df[LABEL_COL].values
    hb3, fp3 = np.digitize(hb, [5.7, 6.5]), np.digitize(fpg, [100, 126])
    agree = pd.DataFrame({"metric": ["Cohen kappa, HbA1c vs fasting-glucose ADA category (3 levels)", "Exact agreement (%)"],
                          "value": [cohen_kappa_score(hb3, fp3), 100 * (hb3 == fp3).mean()]})
    _show(agree.round(3), "Agreement between two ADA laboratory criteria")
    dx = df[OUTCOME_COL].values; meets = (hb >= 6.5) | (fpg >= 126)
    unexpl = int(((dx == 1) & ~meets).sum())
    print(f"Diagnosed patients meeting NO laboratory criterion (HbA1c >= 6.5% or FPG >= 126): {unexpl} "
          f"({100*unexpl/len(df):.1f}% of cohort; {100*unexpl/dx.sum():.1f}% of diagnosed). No medication variable is available to explain them.")
    print(f"Non-diagnosed patients meeting a laboratory criterion: {int(((dx == 0) & meets).sum())}")

    prev = None
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False, "savefig.dpi": 400})
    fig, axs = plt.subplots(1, 3, figsize=(15, 4.2)); x = np.arange(4); names = ["Low", "Moderate", "High", "Very High"]
    w = 0.6
    def bars(ax, t, off, color, label):
        ax.bar(x + off, t["diagnosed diabetes (%)"], w, color=color, edgecolor="black", lw=0.5, label=label,
               yerr=[np.clip(t["diagnosed diabetes (%)"] - t["95% CI low"], 0, None), np.clip(t["95% CI high"] - t["diagnosed diabetes (%)"], 0, None)], capsize=3)
    bars(axs[0], tab, 0, "#3182bd", "HbA1c (ADA) categories")
    axs[0].set_xticks(x); axs[0].set_xticklabels(names); axs[0].set_ylabel("Diagnosed diabetes (%)"); axs[0].set_ylim(0, 110)
    axs[0].set_title("(a) Diagnosis prevalence"); axs[0].legend(frameon=False, fontsize=8, loc="upper left")
    for i, (m, s_) in enumerate(zip(tab["mean fasting glucose (mg/dL)"], [df["glucose_fasting"][lab == c].std() for c in range(4)])):
        axs[1].bar(i, m, 0.6, yerr=s_, capsize=3, color="#3182bd", edgecolor="black", lw=0.5)
    axs[1].axhline(100, color="grey", ls=":", lw=1); axs[1].axhline(126, color="grey", ls="--", lw=1)
    axs[1].set_xticks(x); axs[1].set_xticklabels(names); axs[1].set_ylabel("Fasting glucose, mg/dL (mean ± SD)")
    axs[1].set_title("(b) Fasting glucose (ADA 100 / 126 lines)")
    axs[2].boxplot([df["hba1c"][lab == c] for c in range(4)], showfliers=False, patch_artist=True,
                   boxprops=dict(facecolor="#c6dbef"), medianprops=dict(color="black"))
    axs[2].axhline(5.7, color="grey", ls=":", lw=1); axs[2].axhline(6.5, color="grey", ls="--", lw=1); axs[2].axhline(8.0, color="grey", ls="-.", lw=1)
    axs[2].set_xticklabels(names); axs[2].set_ylabel("HbA1c (%)"); axs[2].set_title("(c) HbA1c by category")
    fp = out / "figures_400dpi"; fp.mkdir(parents=True, exist_ok=True); path = fp / "Fig9_clinical_validation_of_categories.png"
    fig.tight_layout(); fig.savefig(path, dpi=400, bbox_inches="tight"); plt.close(fig); _show_figs([path])

    xlsx = out / VAL_FILE
    with pd.ExcelWriter(xlsx, engine="openpyxl") as xw:
        tab.to_excel(xw, sheet_name="TableA_clinical_categories", index=False)
        pd.DataFrame([ex]).to_excel(xw, sheet_name="Monotonic_association", index=False)
        agree.to_excel(xw, sheet_name="HbA1c_vs_FPG_agreement", index=False)
        try:
            from openpyxl.drawing.image import Image as XLImage
            img = XLImage(str(path)); k = 1000 / img.width; img.width, img.height = 1000, int(img.height * k)
            xw.book.create_sheet("Figure").add_image(img, "A1")
        except Exception as e: print("Figure not embedded:", e)
    print("\nSaved single validation file:", xlsx)


run_clinical_validation()


## 6) SHAP for the 18-PC network (Figures 10 and 11)

In [ ]:
# =====================================================================================
# CELL 5 -- SHAP (Figures 10 and 11): network on the 18 PCs only (no WBM input).   Run AFTER Cell 2.
# SHAP_POLICY: "A"/"B" = expert-based ADA HbA1c labels.
# =====================================================================================
import time
SHAP_DATA_NAME = "diabetes_dataset updated"; SHAP_DATA_PATH = None
SHAP_POLICY = "A"
SHAP_OUT_DIR = "/content/drive/MyDrive/HNFAS_results"; SHAP_FILE = "HNFAS_SHAP_18PC.xlsx"
N_EXPLAIN, N_BACKGROUND = 150, 30            # 150 test patients; background = random sample of 30 training patients
SHAP_EPOCHS, SHAP_BATCH = 60, 256            # manuscript settings: (150, 32)
_NICE = {"hba1c": "HbA1c", "glucose_postprandial": "Post-prandial glucose", "glucose_fasting": "Fasting glucose", "bmi": "BMI",
         "waist_to_hip_ratio": "Waist-hip ratio", "family_history_diabetes": "Family history", "systolic_bp": "Systolic BP",
         "diastolic_bp": "Diastolic BP", "hdl_cholesterol": "HDL", "ldl_cholesterol": "LDL", "cholesterol_total": "Total cholesterol"}
_nice = lambda c: _NICE.get(c, c.replace("_", " ").capitalize())


def run_shap_18pc():
    import shap, matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    seed_everything()
    D = prepare(resolve_data_path(SHAP_DATA_PATH, SHAP_DATA_NAME), SHAP_POLICY)
    P, y, tr, va, te, feats = D["P"], D["lab"], D["tr"], D["va"], D["te"], D["info"]["features"]
    K = P.shape[1]; names = []
    for k in range(K):
        o = np.argsort(-np.abs(D["pca"].components_[k]))[:2]
        names.append(f"PC{k+1} ({_nice(feats[o[0]])}, {_nice(feats[o[1]])})")
    sc = StandardScaler().fit(P[tr]); Xs = sc.transform(P)
    t0 = time.time(); model = build_and_train(Xs[tr], y[tr], Xs[va], y[va], SEED, SHAP_EPOCHS, SHAP_BATCH)
    pred = model.predict(Xs[te], verbose=0).argmax(1)
    print(f"Policy {SHAP_POLICY}: network on {K} PCs trained ({(time.time()-t0)/60:.1f} min); "
          f"test accuracy {accuracy_score(y[te], pred):.4f}, balanced accuracy {balanced_accuracy_score(y[te], pred):.4f}")
    rng = np.random.RandomState(SEED); ex = rng.choice(te, size=min(N_EXPLAIN, len(te)), replace=False)
    bg = Xs[rng.choice(tr, size=min(N_BACKGROUND, len(tr)), replace=False)]
    print(f"Kernel SHAP: {len(ex)} test patients, {N_BACKGROUND} random training patients as background ... (a few minutes)")
    sv = shap.KernelExplainer(lambda A: model.predict(A, verbose=0), bg).shap_values(Xs[ex], silent=True)
    sv = np.stack(sv, axis=-1) if isinstance(sv, list) else np.asarray(sv)
    if sv.shape[0] == 4 and sv.shape[1] == len(ex): sv = np.transpose(sv, (1, 2, 0))      # -> (n, features, classes)
    per_class = np.abs(sv).mean(0); imp = per_class.mean(1); order = np.argsort(-imp)
    tab = pd.DataFrame({"input": [names[i] for i in order], "mean_abs_SHAP (avg of 4 classes)": imp[order],
                        **{f"class {c}": per_class[order, c] for c in range(4)}})
    _show(tab.head(10).round(4), "Mean absolute SHAP values (top 10 inputs)")
    vh = sv[:, :, 3]; dirn = [spearmanr(Xs[ex][:, k], vh[:, k])[0] for k in range(K)]
    print(f"Very High class, Spearman(input value, SHAP): {names[order[0]]} {dirn[order[0]]:+.2f}; {names[order[1]]} {dirn[order[1]]:+.2f}")
    print(f"Top-2 mean|SHAP|: {imp[order[0]]:.3f}, {imp[order[1]]:.3f}; largest among all remaining inputs: {imp[order[2]]:.3f}")

    out = pathlib.Path(SHAP_OUT_DIR)
    try: out.mkdir(parents=True, exist_ok=True)
    except Exception: out = pathlib.Path("/content/HNFAS_results"); out.mkdir(parents=True, exist_ok=True)
    fd = out / "figures_400dpi"; fd.mkdir(parents=True, exist_ok=True)
    plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False})
    fig, ax = plt.subplots(figsize=(9.5, 7)); yy = np.arange(K)[::-1]
    ax.barh(yy, imp[order], color=["#c0392b" if i < 2 else "#5b8db8" for i in range(K)], edgecolor="black", lw=0.5)
    for yv, v in zip(yy, imp[order]): ax.text(v + imp.max() * 0.01, yv, f"{v:.3f}", va="center", fontsize=9)
    ax.set_yticks(yy); ax.set_yticklabels([names[i] for i in order], fontsize=10)
    ax.set_xlabel("Mean |SHAP value| (averaged over the four risk classes)"); ax.set_xlim(0, imp.max() * 1.15)
    p10 = fd / f"Fig10_mean_abs_SHAP_{K}PCs_policy{SHAP_POLICY}.png"; fig.tight_layout(); fig.savefig(p10, dpi=400, bbox_inches="tight"); plt.close(fig)
    shap.summary_plot(vh, Xs[ex], feature_names=names, max_display=K, show=False, plot_size=(10, 7.5))
    fig = plt.gcf(); fig.axes[0].set_xlabel("SHAP value (impact on the Very High-risk output)")
    p11 = fd / f"Fig11_SHAP_beeswarm_VeryHigh_policy{SHAP_POLICY}.png"; fig.savefig(p11, dpi=400, bbox_inches="tight"); plt.close(fig)
    _show_figs([p10, p11], width=800)
    with pd.ExcelWriter(out / SHAP_FILE, engine="openpyxl") as xw:
        tab.to_excel(xw, sheet_name="SHAP_mean_abs", index=False)
        pd.DataFrame({"input": names, "spearman_value_vs_SHAP_VeryHigh": dirn}).to_excel(xw, sheet_name="VeryHigh_direction", index=False)
        ld = []
        for k in range(K):
            o3 = np.argsort(-np.abs(D["pca"].components_[k]))[:3]
            ld.append({"PC": f"PC{k+1}", **{f"top{j+1}": f"{feats[o3[j]]} ({D['pca'].components_[k][o3[j]]:.2f})" for j in range(3)}})
        pd.DataFrame(ld).to_excel(xw, sheet_name="PC_loadings", index=False)
    print("Saved:", out / SHAP_FILE)


run_shap_18pc()


## 7) Consistency of feature importance across models (Figure 12)

In [ ]:
# =====================================================================================
# CELL 6 -- consistency of feature importance across models (Figure 12).   Run AFTER Cell 2.
# Permutation importance on the ORIGINAL variables (complete pipeline: standardisation -> PCA -> classifier),
# drop in balanced accuracy on the held-out test set; agreement by Spearman, Kendall's W (bootstrap CI), top-5 overlap.
# =====================================================================================
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier = None

PERM_DATA_NAME = "diabetes_dataset updated"; PERM_DATA_PATH = None
PERM_POLICY = "A"                      # "A"/"B" = expert-based ADA labels
PERM_OUT_DIR = "/content/drive/MyDrive/HNFAS_results"; PERM_FILE = "HNFAS_permutation_importance.xlsx"
PERM_REPEATS, PERM_TEST_CAP, SVM_TRAIN_CAP, N_BOOT = 5, 5000, 10000, 1000
PERM_EPOCHS, PERM_BATCH = 40, 256          # HNFAS network of this analysis: at most 40 epochs
GLYCAEMIC = ["hba1c", "glucose_fasting", "glucose_postprandial", "insulin_level"]
_NICE = {"hba1c": "HbA1c", "glucose_postprandial": "Post-prandial glucose", "glucose_fasting": "Fasting glucose", "bmi": "BMI",
         "waist_to_hip_ratio": "Waist-hip ratio", "family_history_diabetes": "Family history", "systolic_bp": "Systolic BP",
         "diastolic_bp": "Diastolic BP", "hdl_cholesterol": "HDL", "ldl_cholesterol": "LDL", "cholesterol_total": "Total cholesterol"}
_nice = lambda c: _NICE.get(c, c.replace("_", " ").capitalize())


def _kendall_w(M):
    R = np.apply_along_axis(rankdata, 1, M); m, n = R.shape
    S = ((R.sum(0) - R.sum(0).mean()) ** 2).sum()
    return 12 * S / (m ** 2 * (n ** 3 - n))


def _perm_setting(path, policy, extra_drop, seed=SEED):
    D = prepare(path, policy, extra_drop); P, y, tr, va, te = D["P"], D["lab"], D["tr"], D["va"], D["te"]
    rng = np.random.RandomState(seed); te_s = rng.choice(te, size=min(PERM_TEST_CAP, len(te)), replace=False)
    Xte, yte, feats = D["X"].iloc[te_s].copy(), y[te_s], list(D["X"].columns)
    tr_svm = rng.choice(tr, size=min(SVM_TRAIN_CAP, len(tr)), replace=False)
    sc = StandardScaler().fit(P[tr]); models = {}
    def sk(name, m, idx=tr):
        s_ = StandardScaler().fit(P[idx]); m.fit(s_.transform(P[idx]), y[idx]); models[name] = lambda A, m=m, s_=s_: m.predict(s_.transform(A))
    sk("LR", LogisticRegression(max_iter=500, class_weight="balanced"))
    sk("RF", RandomForestClassifier(n_estimators=200, min_samples_leaf=5, n_jobs=-1, class_weight="balanced_subsample", random_state=seed))
    if XGBClassifier is not None:
        sk("XGBoost", XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, n_jobs=-1, random_state=seed, verbosity=0))
    else:
        sk("XGBoost*", HistGradientBoostingClassifier(max_iter=200, random_state=seed))     # *fallback when xgboost is unavailable
    sk("SVM", SVC(kernel="rbf", C=10, gamma="scale", class_weight="balanced"), idx=tr_svm)
    nn = build_and_train(sc.transform(P[tr]), y[tr], sc.transform(P[va]), y[va], seed, PERM_EPOCHS, PERM_BATCH)
    models["HNFAS"] = lambda A: nn.predict(sc.transform(A), verbose=0).argmax(1)
    pipe = lambda Xd, f: f(D["pca"].transform(D["scaler"].transform(Xd)))
    imp = {}
    for name, f in models.items():
        base = balanced_accuracy_score(yte, pipe(Xte, f)); vals = []
        for col in feats:
            drops = []
            for r in range(PERM_REPEATS):
                Xp = Xte.copy(); Xp[col] = rng.permutation(Xp[col].values)
                drops.append(base - balanced_accuracy_score(yte, pipe(Xp, f)))
            vals.append(np.mean(drops))
        imp[name] = np.array(vals); print(f"   {name:9s} baseline balanced accuracy {base:.3f}")
    I = pd.DataFrame(imp, index=feats); I = I.clip(lower=0); In = I / I.max()
    M = In.values.T; m, n = M.shape; models_l = list(In.columns)
    rho = pd.DataFrame(np.corrcoef(np.apply_along_axis(rankdata, 1, M)), index=models_l, columns=models_l)
    W = _kendall_w(M); rb = np.random.RandomState(seed)
    boots = [_kendall_w(M[:, rb.randint(0, n, n)]) for _ in range(N_BOOT)]
    top5 = [set(np.argsort(-M[i])[:5]) for i in range(m)]
    ov = np.mean([len(top5[i] & top5[j]) / 5 for i in range(m) for j in range(i + 1, m)])
    off = rho.values[np.triu_indices(m, 1)].mean()
    return dict(imp=In, rho=rho, W=W, W_ci=(np.percentile(boots, 2.5), np.percentile(boots, 97.5)), overlap=ov, mean_rho=off, n_vars=n,
                top3={k: list(In[k].sort_values(ascending=False).index[:3]) for k in models_l})


def run_permutation_consistency():
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    seed_everything(); path = resolve_data_path(PERM_DATA_PATH, PERM_DATA_NAME); res = {}
    for key, ed in [("all variables", ()), ("glycaemic variables withheld", tuple(GLYCAEMIC))]:
        print(f"\n== Setting: {key} (policy {PERM_POLICY}) =="); res[key] = _perm_setting(path, PERM_POLICY, ed)
        r = res[key]
        print(f"{r['n_vars']} variables | Kendall W = {r['W']:.2f} (95% CI {r['W_ci'][0]:.2f}-{r['W_ci'][1]:.2f}) | mean top-5 overlap = {r['overlap']:.2f} | mean pairwise Spearman = {r['mean_rho']:.2f}")
        print("top-3 per model:", {k: [_nice(v) for v in vv] for k, vv in r["top3"].items()})
        _show(r["imp"].sort_values(r["imp"].columns[-1], ascending=False).head(10).round(3), f"Normalised permutation importance (top 10) - {key}")
    plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False})
    fig, axs = plt.subplots(2, 2, figsize=(12, 9.5))
    for j, (key, r) in enumerate(res.items()):
        top = r["imp"].loc[r["imp"].mean(1).sort_values(ascending=False).index[:10]]
        ax = axs[0][j]; im = ax.imshow(top.values, cmap="Reds", vmin=0, vmax=1, aspect="auto")
        ax.set_xticks(range(top.shape[1])); ax.set_xticklabels(top.columns); ax.set_yticks(range(len(top))); ax.set_yticklabels([_nice(v) for v in top.index])
        for a in range(top.shape[0]):
            for b in range(top.shape[1]): ax.text(b, a, f"{top.values[a, b]:.2f}", ha="center", va="center", fontsize=7, color="white" if top.values[a, b] > 0.6 else "black")
        ax.set_title(f"({'ab'[j]}) Permutation importance, {key}"); fig.colorbar(im, ax=ax, fraction=0.046)
        ax = axs[1][j]; R_ = r["rho"]; im = ax.imshow(R_.values, cmap="Blues", vmin=-0.2, vmax=1)
        ax.set_xticks(range(len(R_))); ax.set_xticklabels(R_.columns); ax.set_yticks(range(len(R_))); ax.set_yticklabels(R_.index)
        for a in range(len(R_)):
            for b in range(len(R_)): ax.text(b, a, f"{R_.values[a, b]:.2f}", ha="center", va="center", fontsize=8, color="white" if R_.values[a, b] > 0.6 else "black")
        ax.set_title(f"({'cd'[j]}) Spearman between models; W = {r['W']:.2f} [{r['W_ci'][0]:.2f}, {r['W_ci'][1]:.2f}], top-5 overlap {r['overlap']:.2f}", fontsize=8.5)
        fig.colorbar(im, ax=ax, fraction=0.046)
    out = pathlib.Path(PERM_OUT_DIR)
    try: out.mkdir(parents=True, exist_ok=True)
    except Exception: out = pathlib.Path("/content/HNFAS_results"); out.mkdir(parents=True, exist_ok=True)
    fd = out / "figures_400dpi"; fd.mkdir(parents=True, exist_ok=True); p12 = fd / "Fig12_importance_consistency.png"
    fig.tight_layout(); fig.savefig(p12, dpi=400, bbox_inches="tight"); plt.close(fig); _show_figs([p12], width=900)
    with pd.ExcelWriter(out / PERM_FILE, engine="openpyxl") as xw:
        for key, r in res.items():
            r["imp"].to_excel(xw, sheet_name=key[:20] + "_imp"); r["rho"].to_excel(xw, sheet_name=key[:20] + "_rho")
        pd.DataFrame([{"setting": k, "n_variables": r["n_vars"], "kendall_W": r["W"], "W_ci_low": r["W_ci"][0], "W_ci_high": r["W_ci"][1],
                       "top5_overlap": r["overlap"], "mean_pairwise_spearman": r["mean_rho"]} for k, r in res.items()]).to_excel(xw, sheet_name="agreement", index=False)
    print("Saved:", out / PERM_FILE)


run_permutation_consistency()


## 8) Sensitivity to the WBM exponents (p, q) and complexity comparison (Figure 15, Table 10a)

In [ ]:
# =====================================================================================
# CELL 7 -- sensitivity to the WBM exponents (p, q)  [Figure 15]  + complexity comparison [Table 10a].  Run AFTER Cell 2.
# ONE explicit grid is used for the figure, the table and the text.
# =====================================================================================
import time as _time
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier = None
PQ_DATA_NAME = "diabetes_dataset updated"; PQ_DATA_PATH = None; PQ_POLICY = "A"
PQ_OUT_DIR = "/content/drive/MyDrive/HNFAS_results"; PQ_FILE = "HNFAS_pq_sensitivity_and_complexity.xlsx"
PQ_PAIRS = [(1, 1), (1, 0), (2, 0), (5, 0), (2, 1), (5, 1), (10, 1)]      # (1,1) = default; the rest are stress-test settings
COMPLEXITY_TRAIN_CAP = 20000


def run_pq_and_complexity():
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    seed_everything(); D = prepare(resolve_data_path(PQ_DATA_PATH, PQ_DATA_NAME), PQ_POLICY)
    P, tr, te, ev = D["P"], D["tr"], D["te"], D["pca"].explained_variance_
    raw, rows = {}, []
    for (p, q) in PQ_PAIRS:
        fz = FuzzyWBM(WC, p, q).fit(P[tr], ev); raw[(p, q)] = fz.raw_score(P[te])
    names = [f"p={p},q={q}" for p, q in PQ_PAIRS]; base = (1, 1)
    C = np.array([[spearmanr(raw[a], raw[b])[0] for b in PQ_PAIRS] for a in PQ_PAIRS])
    for (p, q) in PQ_PAIRS:
        rows.append({"(p,q)": f"({p},{q})", "Spearman rho vs (1,1)": spearmanr(raw[(p, q)], raw[base])[0],
                     "rho with ADA class": spearmanr(raw[(p, q)], D["lab_ada"][te])[0]})
    tab = pd.DataFrame(rows); _show(tab.round(3), "Sensitivity of the WBM score to (p, q)")
    plt.rcParams.update({"font.size": 10})
    fig, ax = plt.subplots(figsize=(8.2, 6.6)); im = ax.imshow(C, cmap="Reds_r", vmin=max(0.0, C.min() - 0.02), vmax=1)
    ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=35, ha="right"); ax.set_yticks(range(len(names))); ax.set_yticklabels(names)
    for a in range(len(names)):
        for b in range(len(names)): ax.text(b, a, f"{C[a, b]:.3f}", ha="center", va="center", fontsize=8, color="white" if C[a, b] < (C.min() + 1) / 2 else "black")
    ax.set_title(f"Rank correlation between (p,q) settings (test set, n = {len(te)})"); fig.colorbar(im, ax=ax, label="Spearman ρ")
    out = pathlib.Path(PQ_OUT_DIR)
    try: out.mkdir(parents=True, exist_ok=True)
    except Exception: out = pathlib.Path("/content/HNFAS_results"); out.mkdir(parents=True, exist_ok=True)
    fd = out / "figures_400dpi"; fd.mkdir(parents=True, exist_ok=True); p15 = fd / "Fig15_pq_rank_correlation.png"
    fig.tight_layout(); fig.savefig(p15, dpi=400, bbox_inches="tight"); plt.close(fig); _show_figs([p15], width=700)

    # ---------------- complexity (fitted on a capped training subsample; times in seconds) ----------------
    rng = np.random.RandomState(SEED); idx = rng.choice(tr, size=min(COMPLEXITY_TRAIN_CAP, len(tr)), replace=False); y = D["lab"]
    s_ = StandardScaler().fit(P[idx]); Xa, Xt = s_.transform(P[idx]), s_.transform(P[te][:5000]); cr = []
    def timed(name, model, desc, size):
        t0 = _time.time(); model.fit(Xa, y[idx]); tf = _time.time() - t0; t0 = _time.time(); model.predict(Xt); tp = (_time.time() - t0) / len(Xt) * 1000
        cr.append({"model": name, "size measure": size(model), "structure": desc, "train time (s)": tf, "predict time (s / 1000 patients)": tp})
    timed("Logistic Regression", LogisticRegression(max_iter=500), "linear, multinomial", lambda m: f"{m.coef_.size + m.intercept_.size} coefficients")
    timed("Random Forest", RandomForestClassifier(n_estimators=200, min_samples_leaf=5, n_jobs=-1, random_state=SEED), "200 trees, min leaf 5",
          lambda m: f"{len(m.estimators_)} trees, mean depth {np.mean([e.get_depth() for e in m.estimators_]):.1f}, {sum(e.tree_.node_count for e in m.estimators_)} nodes")
    if XGBClassifier is not None:
        timed("XGBoost", XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, n_jobs=-1, verbosity=0, random_state=SEED), "200 rounds x 4 classes, depth 6", lambda m: "800 trees")
    timed("SVM (RBF)", SVC(kernel="rbf", C=10, gamma="scale"), "RBF kernel, C=10", lambda m: f"{int(m.n_support_.sum())} support vectors")
    nn = build_and_train(Xa, y[idx], s_.transform(P[D["va"]][:3000]), y[D["va"]][:3000], SEED, 5, 256)
    t0 = _time.time(); nn.predict(Xt, verbose=0); tp = (_time.time() - t0) / len(Xt) * 1000
    cr.append({"model": "HNFAS (18 PCs -> 128-64-32-4 network)", "size measure": f"{nn.count_params():,} parameters (+ PCA {P.shape[1]}x{D['info']['n_features']} projection)",
               "structure": "PCA + Gaussian fuzzification + WBM (labelling/interpretation stage) + DNN classifier", "train time (s)": float("nan"),
               "predict time (s / 1000 patients)": tp})
    ctab = pd.DataFrame(cr); _show(ctab.round(4), "Complexity comparison (Table 10a)")
    with pd.ExcelWriter(out / PQ_FILE, engine="openpyxl") as xw:
        tab.to_excel(xw, sheet_name="pq_sensitivity", index=False); pd.DataFrame(C, index=names, columns=names).to_excel(xw, sheet_name="pq_spearman_matrix")
        ctab.to_excel(xw, sheet_name="complexity", index=False)
    print("Saved:", out / PQ_FILE)


run_pq_and_complexity()
